In [1]:
import pandas as pd
from pathlib import Path

In [2]:
metric = "CPE"

## InfiniumPurify (RefFree)

In [3]:
reffree_dir = Path("../../data/cancer-methyl/infiniumpurify_results/TCGA/all_20260106_184744")
folders = reffree_dir.iterdir()

dfs = []
for folder in folders:
    if folder.is_dir() and folder.name != ".ipynb_checkpoints":
        for file in folder.iterdir():
            if file.name.endswith("refFree_test_purity.parquet"):
                df = pd.read_parquet(file)
                dfs.append(df)
infinium_reffree_purity = pd.concat(dfs, ignore_index=True)
# infinium_reffree_purity["Method"] = "InfiniumPurify (dataset-fitted)"
infinium_reffree_purity.set_index("Barcode", inplace=True)

## InfiniumPurify (refBased)

In [4]:
refbased_dir = Path("../../data/cancer-methyl/infiniumpurify_results/TCGA/refBased_20251113_113223/")
folders = refbased_dir.iterdir()

dfs = []
for folder in folders:
    cancer_type = folder.name
    if folder.is_dir() and folder.name != ".ipynb_checkpoints":
        for file in folder.iterdir():
            if file.name.endswith(f"{cancer_type}_purity.parquet"):
                df = pd.read_parquet(file)
                dfs.append(df)
infinium_refbased_purity = pd.concat(dfs, ignore_index=True)
# infinium_refbased_purity["Method"] = "InfiniumPurify (pretrained)"
infinium_refbased_purity.set_index("Barcode", inplace=True)
# infinium_refbased_purity = infinium_refbased_purity.loc[infinium_reffree_purity.index]

## PureBeta (refFree)

In [5]:
reffree_dir = Path("../../data/cancer-methyl/purebeta_results/final_split_trainval_to_test/20260113_123302/")
folders = reffree_dir.iterdir()

dfs = []
for folder in folders:
    if folder.is_dir() and folder.name != ".ipynb_checkpoints":
        for file in folder.iterdir():
            if file.name.endswith("test_purity_estimates.parquet"):
                df = pd.read_parquet(file)
                dfs.append(df)
purebeta_reffree_purity = pd.concat(dfs, ignore_index=True)
# purebeta_reffree_purity["Method"] = "PureBeta (dataset-fitted)"
purebeta_reffree_purity.set_index("Barcode", inplace=True)


## PAMES

In [6]:
pames_purity = pd.read_csv("../../data/benchmark/PAMES_sample_purity.csv")
pames_purity.set_index("Barcode", inplace=True)

## Merge

In [7]:
merged_purity = (
    pd.concat(
        [
            infinium_reffree_purity.rename(columns={"Purity": "Purity_reffree"}),
            infinium_refbased_purity.rename(columns={"Purity": "Purity_refbased"}),
            purebeta_reffree_purity.rename(columns={"Purity": "Purity_purebeta"}),
            pames_purity.rename(columns={"PAME": "Purity_pames"}),
        ],
        axis=1,
        join="outer",
    )
)

merged_purity.head()

,Purity_reffree,Purity_refbased,Purity_purebeta,Sample,Patient,SampleTypeCode,SampleType,Cancer.type,ESTIMATE,ABSOLUTE,LUMP,IHC,CPE,Project,T_N,Cancer_T_N,split,CPE-AEI,CPE-AE,Purity_pames
Barcode,,,,,,,,,,,,,,,,,,,,
TCGA-H4-A2HO-01A-11D-A17Y-05,0.683976,0.651015,NaN,TCGA-H4-A2HO-01A,TCGA-H4-A2HO,1.0,Primary Tumor,BLCA,0.8986,0.64,0.7135,0.75,0.7339,TCGA-BLCA,Tumor,TCGA-BLCA_Tumor,test,0.817488,0.838366,0.706866
TCGA-FD-A43P-01A-31D-A23V-05,0.573329,0.529187,NaN,TCGA-FD-A43P-01A,TCGA-FD-A43P,1.0,Primary Tumor,BLCA,0.7415,NaN,0.6379,0.90,0.6595,TCGA-BLCA,Tumor,TCGA-BLCA_Tumor,test,0.804509,NaN,0.710182
TCGA-G2-A3VY-01A-11D-A231-05,0.883125,0.859811,NaN,TCGA-G2-A3VY-01A,TCGA-G2-A3VY,1.0,Primary Tumor,BLCA,0.9955,NaN,0.9761,0.90,1.0000,TCGA-BLCA,Tumor,TCGA-BLCA_Tumor,train,0.954598,NaN,0.909892
TCGA-K4-A6FZ-01A-11D-A31M-05,0.538611,0.512542,NaN,TCGA-K4-A6FZ-01A,TCGA-K4-A6FZ,1.0,Primary Tumor,BLCA,0.8766,NaN,0.6405,0.70,0.8430,TCGA-BLCA,Tumor,TCGA-BLCA_Tumor,train,0.739224,NaN,0.620469
TCGA-FD-A3B8-01A-31D-A211-05,0.323989,0.282930,NaN,TCGA-FD-A3B8-01A,TCGA-FD-A3B8,1.0,Primary Tumor,BLCA,0.3916,0.18,0.3541,0.85,0.3047,TCGA-BLCA,Tumor,TCGA-BLCA_Tumor,train,0.449102,0.267439,0.452362


In [8]:
merged_purity = merged_purity.loc[:, ["Purity_reffree", "Purity_refbased", "Purity_purebeta", "Purity_pames"]]

In [10]:
merged_purity.columns = ["InfiniumPurify (dataset-fitted)", "InfiniumPurify (pretrained)", "PureBeta (dataset-fitted)", "PAMES"]

In [11]:
merged_purity.to_csv("../../data/benchmark/all_methods_purity_estimates.csv")

Owing to PurbeBeta (pretrained) saved in RData, please check the code in `src/scripts/merge_purebeta_pretrained_correlation.R`

## Calcualte the correlation

In [12]:
df = pd.read_csv("../../data/benchmark/all_methods_purity_estimates.csv", index_col=0)

In [13]:
monte = pd.read_csv("../../data/monte_outputs/pancancer/monte_pancancer_meta_with_predictions_test.csv")
monte.set_index("Barcode", inplace=True)

In [14]:
monte = monte.loc[:, ["Cancer.type", metric, f"predicted_{metric}"]]

In [15]:
monte.rename(columns={f"predicted_{metric}": "MONTE"}, inplace=True)

In [16]:
df = pd.merge(df, monte[["Cancer.type", metric, "MONTE"]], left_index=True, right_index=True)

In [17]:
df = df.dropna(subset=[metric])

In [18]:
def safe_corr(g, col):
    if col not in g.columns:
        return float("nan")
    valid = g[[metric, col]].dropna()
    return valid[metric].corr(valid[col]) if len(valid) > 1 else float("nan")


corr_by_cancer = df.groupby("Cancer.type").apply(
    lambda g: pd.Series(
        {
            "n_samples": len(g),
            "InfiniumPurify (dataset-fitted)": safe_corr(g, "InfiniumPurify (dataset-fitted)"),
            "InfiniumPurify (pretrained)": safe_corr(g, "InfiniumPurify (pretrained)"),
            "PureBeta (dataset-fitted)": safe_corr(g, "PureBeta (dataset-fitted)"),
            "PureBeta (pretrained)": safe_corr(g, "PureBeta (pretrained)"),
            "PAMES": safe_corr(g, "PAMES"),
            "MONTE": safe_corr(g, "MONTE"),
        }
    )
)

corr_by_cancer.to_csv("../../data/benchmark/benchmark_all_methods_purity_correlation_by_cancer.csv")